# 12 · Selected-Area Electron Diffraction (SAED) Workflows

**Where this sits in PyTex.** Notebook 08 built the diffraction geometry and a single
kinematic spot pattern; Notebook 11 did powder XRD. Here we generate a full
**single-crystal SAED** pattern for a chosen **zone axis**, the everyday tool of TEM
crystallography, with calibrated detector coordinates and selection-rule-aware
intensities.

## Learning goals

1. generate a SAED pattern with `generate_saed_pattern` down a chosen zone axis;
2. verify every spot obeys the **zone law** $hu+kv+lw=0$;
3. verify the **camera-constant** calibration $R = C\,|\mathbf g| = C/d$;
4. see the **FCC selection rule** as zero-intensity forbidden spots;
5. compare the symmetry of the $[001]$ and $[011]$ patterns.

## Theory

A near-parallel electron beam down a zone axis $[uvw]$ illuminates the
zero-order Laue zone: the reflections $\{hkl\}$ whose reciprocal-lattice vectors are
perpendicular to the beam, i.e. those satisfying the **zone law**
$hu+kv+lw=0$. Because the Ewald sphere is almost flat for fast electrons (Notebook 08),
a whole planar section of reciprocal space is excited at once, producing a 2-D lattice
of spots.

The distance $R$ of a spot from the centre relates to the plane spacing through the
**camera constant** $C = L\lambda$:

$$
R \, d_{hkl} = L\lambda = C \quad\Longleftrightarrow\quad R = C\,|\mathbf g_{hkl}| .
$$

The *intensity* still follows the structure factor, so kinematically forbidden
reflections (mixed-parity $hkl$ for FCC) appear at zero intensity. (Williams & Carter
2009; De Graef 2003.)

In [ ]:
from __future__ import annotations

import warnings

import numpy as np

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    ReferenceFrame,
    ZoneAxis,
    generate_saed_pattern,
    get_phase_fixture,
    plot_saed_pattern,
)

np.set_printoptions(precision=4, suppress=True)

crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
nickel = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)

## 1 · A pattern down the $[001]$ zone

`generate_saed_pattern` needs the phase, a `ZoneAxis`, and a **camera constant**
(mm·Å). The returned `SAEDPattern` lists the spots with their crystal reciprocal
vectors, calibrated detector coordinates, intensities, and text labels.

In [ ]:
camera_constant = 180.0  # mm * angstrom
zone_001 = ZoneAxis(indices=np.array([0, 0, 1]), phase=nickel)
saed_001 = generate_saed_pattern(
    nickel, zone_001, camera_constant_mm_angstrom=camera_constant,
    max_index=3, max_g_inv_angstrom=3.0,
)
print("zone axis        :", tuple(int(x) for x in saed_001.zone_axis.indices))
print("spots generated  :", len(saed_001.spots))
print("detector frame   :", saed_001.detector_frame.name)
print("example labels    :", [s.label for s in saed_001.spots if s.label][:5])

## 2 · Every spot obeys the zone law

For the $[001]$ zone the law $h\cdot0+k\cdot0+l\cdot1=0$ requires $l=0$. We confirm it
for all spots — this is what makes a SAED pattern a slice of reciprocal space.

In [ ]:
miller = np.array([s.miller_indices for s in saed_001.spots])
zone_dot = miller @ np.array([0, 0, 1])
print("all spots satisfy the zone law (l = 0):", bool(np.all(zone_dot == 0)))
assert np.all(zone_dot == 0)

## 3 · Camera-constant calibration $R = C\,|\mathbf g|$

The whole point of a camera constant is that a measured spot radius $R$ converts to a
$d$-spacing. We verify the exact relation $R = C\,|\mathbf g_{hkl}|$ across all spots:
dividing each spot's detector radius by its reciprocal-vector length returns the camera
constant we asked for.

In [ ]:
radii = np.array([np.linalg.norm(s.detector_coordinates) for s in saed_001.spots])
g_len = np.array([np.linalg.norm(s.reciprocal_vector_crystal) for s in saed_001.spots])
nonzero = g_len > 0
recovered = radii[nonzero] / g_len[nonzero]
print(f"recovered camera constant (mm*A): min {recovered.min():.4f}, max {recovered.max():.4f}")
assert np.allclose(recovered, camera_constant, atol=1e-6)

## 4 · Selection rules: forbidden spots carry zero intensity

The generator places every geometrically allowed spot of the zone, but the **structure
factor** decides which are visible. For FCC, only unmixed-parity $hkl$ scatter; the
mixed-parity spots (e.g. $100$, $110$) are present in the list with **exactly zero
intensity**. This is the reciprocal-space view of the same selection rule seen as
missing powder lines in Notebook 11.

In [ ]:
intensity = np.array([s.intensity for s in saed_001.spots])
unmixed = (miller[:, 0] % 2 == miller[:, 1] % 2) & (miller[:, 1] % 2 == miller[:, 2] % 2)
print(f"allowed (unmixed parity) spots : {int(unmixed.sum())}, all with intensity > 0: "
      f"{bool(np.all(intensity[unmixed] > 0))}")
print(f"forbidden (mixed parity) spots : {int((~unmixed).sum())}, max intensity: "
      f"{intensity[~unmixed].max():.3e}")
assert np.all(intensity[unmixed] > 0)
assert np.allclose(intensity[~unmixed], 0.0)

## 5 · Two zones, two symmetries

The visible-spot symmetry reflects the projected reciprocal lattice: the $[001]$ zone
of FCC shows a **square** array (the $\{200\}$/$\{220\}$ spots), while the $[011]$ zone
shows a **centred-rectangular** array. Plotting both makes the difference obvious — and
this pattern-recognition is exactly how a microscopist identifies orientation.

In [ ]:
import matplotlib.pyplot as plt

zone_011 = ZoneAxis(indices=np.array([0, 1, 1]), phase=nickel)
saed_011 = generate_saed_pattern(
    nickel, zone_011, camera_constant_mm_angstrom=camera_constant,
    max_index=3, max_g_inv_angstrom=3.0,
)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.8))
plot_saed_pattern(saed_001, theme="dark", ax=axes[0])
axes[0].set_title(r"Ni FCC SAED — $[001]$ zone (square)")
plot_saed_pattern(saed_011, theme="dark", ax=axes[1])
axes[1].set_title(r"Ni FCC SAED — $[011]$ zone")
fig.tight_layout()


## Summary and where to go next

- `generate_saed_pattern` produces a calibrated single-crystal spot pattern for a zone axis.
- Every spot obeys the **zone law**; the **camera constant** links spot radius to
  $d$-spacing via $R = C|\mathbf g|$.
- **FCC selection rules** appear as zero-intensity forbidden spots; different zone axes
  show different spot symmetries.

**Next:** [Notebook 21](21_composite_or_diffraction_patterns.ipynb) overlays parent and
transformation-variant SAED patterns for orientation-relationship analysis, and
[Notebook 15](15_structure_diffraction_visualization_pipeline.ipynb) embeds SAED in a
manifest-tracked structure→diffraction pipeline.